In [1]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal,Independent
import gym

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
class Config:
    
    ENV_NAME = "Ant-v4"
    
    N_LAYERS = 2          # 隐藏层数量
    HIDDEN_SIZE = 64      # 每层隐藏单元数
    LEARNING_RATE = 5e-3  # 学习率
    
    # 训练配置
    BATCH_SIZE = 100      # 训练 batch size
    N_EPOCHS = 100        # 训练轮数
    
        
    # 数据路径
    EXPERT_DATA_PATH = "cs224r/expert_data/expert_data_Ant-v4.pkl"
    
    # 设备
    DEVICE = torch.device("mps" if torch.backends.mps.is_available() else 
                         "cuda" if torch.cuda.is_available() else "cpu")

In [36]:
class MLPPolicy(nn.Module):
    """
    多层感知机策略网络
    输入：观测 (obs_dim,)
    输出：动作分布 (高斯分布，均值 + 标准差)
    """
    
    def __init__(self, obs_dim, ac_dim, n_layers=2, hidden_size=64):
        super().__init__()
        
        # 构建 MLP：obs_dim -> hidden_size -> ... -> hidden_size -> ac_dim
        layers = []
        
        # 第一层：输入层 -> 隐藏层
        layers.append(nn.Linear(obs_dim, hidden_size))
        layers.append(nn.Tanh())
        
        # 中间隐藏层
        for _ in range(n_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.Tanh())
        
        # 输出层：隐藏层 -> 动作维度（输出均值）
        layers.append(nn.Linear(hidden_size, ac_dim))
        
        self.mean_net = nn.Sequential(*layers)
        
        # 标准差参数（可学习的标量，广播到所有动作维度）
        self.log_std = nn.Parameter(torch.zeros(ac_dim))
        
    def forward(self, obs):
        """
        前向传播：给定观测，返回动作分布
        
        Args:
            obs: [batch_size, obs_dim] 或 [obs_dim]
        
        Returns:
            dist: Independent(Normal) 分布对象
        """
        # 如果是单个观测，添加 batch 维度
        if len(obs.shape) == 1:
            obs = obs.unsqueeze(0)
        
        # 计算均值：[batch_size, ac_dim]
        mean = self.mean_net(obs)
        
        # 计算标准差：[batch_size, ac_dim]（广播）
        std = torch.exp(self.log_std.expand_as(mean))
        
        # 创建高斯分布（每个动作维度独立）
        # Independent 将 ac_dim 个独立的一维高斯分布组合成一个分布
        dist = Independent(Normal(mean, std), 1)
        
        return dist
    
    def get_action(self, obs):
        """
        采样动作（用于推理/演示）
        
        Args:
            obs: numpy array [obs_dim] 或 [batch_size, obs_dim]
        
        Returns:
            action: numpy array [ac_dim] 或 [batch_size, ac_dim]
        """
        # 转换为 tensor
        if isinstance(obs, np.ndarray):
            obs_tensor = torch.from_numpy(obs).float().to(Config.DEVICE)
        else:
            obs_tensor = obs.to(Config.DEVICE)
        
        # 采样动作
        with torch.no_grad():
            dist = self.forward(obs_tensor)
            action_tensor = dist.sample()
        
        # 转换回 numpy
        return action_tensor.cpu().numpy()

In [37]:
def load_expert_data(filepath):
    """
    加载专家数据
    
    Args:
        filepath: 专家数据 pickle 文件路径
    
    Returns:
        observations: [N, obs_dim] numpy array
        actions: [N, ac_dim] numpy array
    """
    print(f"Loading expert data from {filepath}...")
    
    with open(filepath, 'rb') as f:
        data = pickle.load(f)
    
    
    # 处理不同的数据格式
    if isinstance(data, dict):
        # 格式 1: dict with 'observations' and 'actions'
        obs = data.get('observations', data.get('obs'))
        acs = data.get('actions', data.get('acs'))
        
        if obs is None or acs is None:
            raise ValueError("Expert data must contain 'observations' and 'actions'")
        
        # 如果是列表，转换为数组
        if isinstance(obs, list):
            obs = np.array(obs)
        if isinstance(acs, list):
            acs = np.array(acs)
        
        # 如果是多条轨迹，展平
        if len(obs.shape) == 3:  # [n_trajs, T, obs_dim]
            obs = obs.reshape(-1, obs.shape[-1])
        if len(acs.shape) == 3:  # [n_trajs, T, ac_dim]
            acs = acs.reshape(-1, acs.shape[-1])
        
        # 确保是 2D
        if len(obs.shape) == 1:
            obs = obs.reshape(1, -1)
        if len(acs.shape) == 1:
            acs = acs.reshape(1, -1)
            
    elif isinstance(data, list):
        # 格式 2: list of trajectories
        obs_list = []
        acs_list = []
        for traj in data:
            if isinstance(traj, dict):
                obs_list.append(traj['observation'])
                acs_list.append(traj['action'])
            else:
                obs_list.append(traj[0])
                acs_list.append(traj[1])
        
        obs = np.concatenate(obs_list, axis=0)
        acs = np.concatenate(acs_list, axis=0)
    else:
        raise ValueError(f"Unsupported data format: {type(data)}")
    
    print(f"Loaded {len(obs)} expert (state, action) pairs")
    print(f"  Observation shape: {obs.shape}")
    print(f"  Action shape: {acs.shape}")
    
    return obs, acs

In [38]:
load_expert_data(Config.EXPERT_DATA_PATH)

Loading expert data from cs224r/expert_data/expert_data_Ant-v4.pkl...
Loaded 2000 expert (state, action) pairs
  Observation shape: (2000, 111)
  Action shape: (2000, 8)


(array([[ 0.68822366,  0.9896817 ,  0.05754574, ...,  0.        ,
          0.        ,  0.        ],
        [ 0.7131519 ,  0.9907151 ,  0.06424748, ...,  0.        ,
          0.        ,  0.        ],
        [ 0.700461  ,  0.9919145 ,  0.07233977, ...,  0.        ,
          0.        ,  0.        ],
        ...,
        [ 0.50590795,  0.96818256,  0.00853824, ...,  0.        ,
          0.        ,  0.        ],
        [ 0.4620163 ,  0.97200114, -0.01413327, ...,  0.        ,
          0.        ,  0.        ],
        [ 0.48443967,  0.9687733 ,  0.00342305, ...,  0.        ,
          0.        ,  0.        ]], dtype=float32),
 array([[ 2.17757374e-01,  1.17660418e-01, -1.07873464e+00, ...,
          1.11716300e-01,  7.17068136e-01,  1.90164894e-01],
        [-5.85895479e-01, -8.61527681e-01,  3.86943430e-01, ...,
          9.09744263e-01, -2.74880379e-01,  1.27099633e+00],
        [ 1.77240483e-02, -7.90676713e-01, -2.42501691e-01, ...,
          4.69385207e-01,  2.89635628e-01

In [39]:
def train_bc(policy,observations,actions,n_epochs=100,batch_size=100):
    
    optimizer = optim.Adam(policy.parameters(),lr=Config.LEARNING_RATE)
    obs_tensor = torch.from_numpy(observations).float().to(Config.DEVICE)
    act_tensor = torch.from_numpy(actions).float().to(Config.DEVICE)
    
    losses = []
    n_batches = len(observations) // batch_size
    
    for epoch in range(n_batches):
        
        epoch_loss = 0.0
        indices = np.random.permutation(len(observations))
        
        for i in range(n_batches):
            
            batch_indices = indices[i*batch_size:(i+1)*batch_size]
            obs_batch = obs_tensor[batch_indices]
            act_batch = act_tensor[batch_indices]
            
            dist = policy(obs_batch)
            
            log_probs = dist.log_prob(act_batch)
            loss = -log_probs.mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(epoch,avg_loss)
        
        
    return losses
        

In [40]:
def evaluate_policy(policy, env, n_episodes=5, max_steps=1000, render=False):
    policy.eval()
    returns = []
    
    episode_lengths = []
    
    for episode in range(n_episodes):
        
        reset_result = env.reset()
        
        if isinstance(reset_result,tuple):
            obs,_ = reset_result
        else:
            obs = reset_result
        
        total_reward = 0
        steps = 0
        
        while steps < max_steps:
            action = policy.get_action(obs)
            
            if len(action.shape) > 1:
                action = action[0]
            
            next_obs,reward,done,info = env.step(action)
            
            if render:
                env.render()
            
            total_reward += reward
            steps += 1
            obs = next_obs
            
            if done:
                break
        
        returns.append(total_reward)
        episode_lengths.append(steps)
        
        print(f"Episode {episode+1}: Return = {total_reward:.2f}, Length = {steps}")
    
    avg_return = np.mean(returns)
    std_return = np.std(returns)
    avg_length = np.mean(episode)
    
        
    print(f"\n{'='*60}")
    print(f"Average Return: {avg_return:.2f} ± {std_return:.2f}")
    print(f"Average Episode Length: {avg_length:.1f}")
    print(f"{'='*60}\n")
    
    return returns, episode_lengths

In [41]:
def main():
    """
    完整的 Behavior Cloning 流程：
    1. 创建环境
    2. 加载专家数据
    3. 创建策略网络
    4. 训练策略
    5. 评估策略
    6. 演示（可选）
    """
    print("="*60)
    print("Behavior Cloning - Minimal Standalone Implementation")
    print("="*60)
    
    # ========================================================================
    # 步骤 1: 创建环境
    # ========================================================================
    print("\n[Step 1] Creating environment...")
    env_kwargs = {"render_mode": "rgb_array"}
    if Config.ENV_NAME == "Ant-v4":
        env_kwargs["use_contact_forces"] = True
    
    env = gym.make(Config.ENV_NAME, **env_kwargs)
    obs_dim = env.observation_space.shape[0]
    ac_dim = env.action_space.shape[0]
    
    print(f"Environment: {Config.ENV_NAME}")
    print(f"Observation dimension: {obs_dim}")
    print(f"Action dimension: {ac_dim}")
    
    # ========================================================================
    # 步骤 2: 加载专家数据
    # ========================================================================
    print("\n[Step 2] Loading expert data...")
    try:
        observations, actions = load_expert_data(Config.EXPERT_DATA_PATH)
    except FileNotFoundError:
        print(f"ERROR: Expert data file not found: {Config.EXPERT_DATA_PATH}")
        print("Please make sure the file exists.")
        return
    
    # ========================================================================
    # 步骤 3: 创建策略网络
    # ========================================================================
    print("\n[Step 3] Creating policy network...")
    policy = MLPPolicy(
        obs_dim=obs_dim,
        ac_dim=ac_dim,
        n_layers=Config.N_LAYERS,
        hidden_size=Config.HIDDEN_SIZE
    )
    policy.to(Config.DEVICE)
    
    # 打印网络结构
    total_params = sum(p.numel() for p in policy.parameters())
    trainable_params = sum(p.numel() for p in policy.parameters() if p.requires_grad)
    print(f"Policy network created:")
    print(f"  Layers: {Config.N_LAYERS} hidden layers")
    print(f"  Hidden size: {Config.HIDDEN_SIZE}")
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    
    # ========================================================================
    # 步骤 4: 训练策略
    # ========================================================================
    print("\n[Step 4] Training policy...")
    losses = train_bc(
        policy=policy,
        observations=observations,
        actions=actions,
        n_epochs=Config.N_EPOCHS,
        batch_size=Config.BATCH_SIZE
    )
    
    # ========================================================================
    # 步骤 5: 评估策略
    # ========================================================================
    print("\n[Step 5] Evaluating trained policy...")
    returns, lengths = evaluate_policy(
        policy=policy,
        env=env,
        n_episodes=5,
        max_steps=1000,
        render=False
    )
    
    # ========================================================================
    # 步骤 6: 保存策略参数
    # ========================================================================
    print("\n[Step 6] Saving trained policy...")
    policy_path = "bc_policy_standalone.pt"
    torch.save(policy.state_dict(), policy_path)
    print(f"Policy saved to: {policy_path}")
    
    # 关闭环境
    env.close()
    
    print("\n" + "="*60)
    print("BC Training Complete!")
    print(f"Policy saved to: {policy_path}")
    print("To visualize the policy, run: python load_and_demo_policy.py")
    print("="*60)




In [42]:
main()

Behavior Cloning - Minimal Standalone Implementation

[Step 1] Creating environment...
Environment: Ant-v4
Observation dimension: 111
Action dimension: 8

[Step 2] Loading expert data...
Loading expert data from cs224r/expert_data/expert_data_Ant-v4.pkl...
Loaded 2000 expert (state, action) pairs
  Observation shape: (2000, 111)
  Action shape: (2000, 8)

[Step 3] Creating policy network...
Policy network created:
  Layers: 2 hidden layers
  Hidden size: 64
  Total parameters: 11,856
  Trainable parameters: 11,856

[Step 4] Training policy...
0 7.15359251499176
9 -0.06125901471823454
19 -7.032101368904113

[Step 5] Evaluating trained policy...


/Users/liuchu/opt/anaconda3/envs/cs224r/lib/python3.11/site-packages/gym/utils/passive_env_checker.py:241: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


Episode 1: Return = 1106.46, Length = 1000
Episode 2: Return = 1146.23, Length = 411
Episode 3: Return = 619.25, Length = 199
Episode 4: Return = -84.80, Length = 1000
Episode 5: Return = 3185.08, Length = 1000

Average Return: 1194.44 ± 1089.95
Average Episode Length: 4.0


[Step 6] Saving trained policy...
Policy saved to: bc_policy_standalone.pt

BC Training Complete!
Policy saved to: bc_policy_standalone.pt
To visualize the policy, run: python load_and_demo_policy.py
